# Flood-Risk Classification for Kenya: Base Model
**ICS 3202 Artificial Intelligence | Project Deliverable 2 (CAT 2)**

Team: Victor Mwakisagu, Eric Ghati, Tashome Makonnen

**AI application:** Classification (binary)  
**Base model:** Logistic Regression (L2-regularised)  
**Comparison:** Random Forest, and two rule-based baselines (50 mm rule; 50 mm + 200% of normal)

## 1. Dataset coverage

In [1]:
import pandas as pd
#________________________________________________________________________________________________________________________
#once again using the CHIRPS rainfall indicator by the UN
#ONE ROW = 1 admin unit for one DEKAD(a fixed 10 day reporting period, about 3 in a month)
#uses 10 day window no t a single day, this matters cause flooding is in hours but the finest time reolution is 10 days
#so ideally cant say which day but rather which 10 day window is at risk
#________________________________________________________________________________________________________________________


df = pd.read_csv("data/ken-rainfall-subnat-full.csv")


#________________________________________________________________________________________________________________________
#adm_level will tell us how coarse each row is:
#   level 1 = county     (Kenya has 47)
#   level 2 = sub-county (Kenya has roughly 290)
#Both levels sit in the same file stacked on top of each other.
#the row and its sub-county row can describe the same rain.
#
#PCODE is the unit's "place code", structures as:
#   KE047    -> country KE, county 047 (Nairobi)
#   KE047277 -> country KE, county 047, sub-county 277
#So the first 5 characters of ANY code tell us its county.
#_________________________________________________________________________________________________________________________


county_units    = df[df.adm_level == 1]["PCODE"].unique() #county-level units
subcounty_units = df[df.adm_level == 2]["PCODE"].unique() #sub-county-level units

print("level-1 units:", len(county_units))
print(sorted(county_units))
print("\nlevel-2 units:", len(subcounty_units))


#___________________________________________________________________________________________________
#coverage check,
# Kenya has 47 counties and this file doesn't enirely cover that
# #this is a partial export and every label we build on it would be incomplete
# slicing the county prefix off each sub-county code will tell us how many counties are represented,
#even where no county-level row exists.
#___________________________________________________________________________________________________
counties = sorted({p[:5] for p in subcounty_units}) #set comprehension for unique prefixes
print("\ndistinct counties appearing in level-2 codes:", len(counties))
print(counties)

#_____________________________________________________________________________________
#How many sub-counties we actually have per county. If this is
#1-2 each rather than all of them we have a sample of each county rather than 
#full coverage which is a limitation
#______________________________________________________________________________________

per = pd.Series([p[:5] for p in subcounty_units]).value_counts().sort_index()
print("\nsub-counties per county:")
print(per.to_string())


#______________________________________________________________________________________________________________
#The 8 counties have an official county-level row and sub-county row
#if aggregating sub-counties reproduces the official figure our aggregation method can be considered valid for
#the other 39 counties.
#_______________________________________________________________________________________________________________


for code in county_units:
    official = df[(df.PCODE == code)][["date", "rfh"]].set_index("date")["rfh"]
    subs = df[(df.adm_level == 2) & (df.PCODE.str.startswith(code))]
    derived = subs.groupby("date")["rfh"].mean()
    diff = (official - derived).abs()
    print(code, "| n_subs:", subs.PCODE.nunique(),
          "| mean abs diff:", round(diff.mean(), 3),
          "| max:", round(diff.max(), 3))


#____________________________________________________________________
# a plain mean treats a small sub-county and a huge one equally.
#n-pixels is how many CHIRPS grid cells each unit covers, so it
#stands in for area where weighting by it should track the official
#county figure more closely
#____________________________________________________________________


for code in county_units:
    official = df[df.PCODE == code][["date", "rfh"]].set_index("date")["rfh"]
    subs = df[(df.adm_level == 2) & (df.PCODE.str.startswith(code))].copy()
    subs["weighted"] = subs["rfh"] * subs["n_pixels"]
    agg = subs.groupby("date").apply(
        lambda g: g["weighted"].sum() / g["n_pixels"].sum()
    )
    diff = (official - agg).abs()
    print(code,"| weighted mean abs diff:", round(diff.mean(), 3),
          "| max:", round(diff.max(), 3) )

level-1 units: 8
['KE004', 'KE008', 'KE010', 'KE019', 'KE023', 'KE037', 'KE043', 'KE047']

level-2 units: 73

distinct counties appearing in level-2 codes: 47
['KE001', 'KE002', 'KE003', 'KE004', 'KE005', 'KE006', 'KE007', 'KE008', 'KE009', 'KE010', 'KE011', 'KE012', 'KE013', 'KE014', 'KE015', 'KE016', 'KE017', 'KE018', 'KE019', 'KE020', 'KE021', 'KE022', 'KE023', 'KE024', 'KE025', 'KE026', 'KE027', 'KE028', 'KE029', 'KE030', 'KE031', 'KE032', 'KE033', 'KE034', 'KE035', 'KE036', 'KE037', 'KE038', 'KE039', 'KE040', 'KE041', 'KE042', 'KE043', 'KE044', 'KE045', 'KE046', 'KE047']

sub-counties per county:
KE001    1
KE002    1
KE003    2
KE004    1
KE005    1
KE006    1
KE007    2
KE008    1
KE009    1
KE010    3
KE011    1
KE012    2
KE013    2
KE014    2
KE015    2
KE016    1
KE017    1
KE018    1
KE019    1
KE020    1
KE021    3
KE022    1
KE023    1
KE024    1
KE025    1
KE026    1
KE027    1
KE028    2
KE029    2
KE030    2
KE031    1
KE032    1
KE033    2
KE034    1
KE035    1
KE036 

KE019 | weighted mean abs diff: 4.277 | max: 70.147
KE004 | weighted mean abs diff: 6.398 | max: 60.246


KE010 | weighted mean abs diff: 5.323 | max: 69.378
KE047 | weighted mean abs diff: 0.0 | max: 0.0


KE008 | weighted mean abs diff: 2.646 | max: 43.126
KE043 | weighted mean abs diff: 4.17 | max: 27.37


KE023 | weighted mean abs diff: 10.802 | max: 53.204
KE037 | weighted mean abs diff: 4.635 | max: 28.885


## 2. Data cleaning and EDA (Section A)

*Tashome's section: see `tashome_cleaning.ipynb`.*

## 3. Flood labels

### 3.1 Matching place names to county codes

In [2]:
import re
import pandas as pd

#_______________________________________________________________________________
#Purpose of this script is as follows:
#the 2 sources naming conventions are messed up in free text
#("TANA-RIVER" OR "Kisumu districts or provinces like nyanza or even mispellings like tarkana")
#Our rainfall data uses official codes like KE042 = Kisumu
#So this script is to ubild a bridge in that any place name -> county code
#Exit test is every name either matches or is intentionally ignored
#________________________________________________________________________________


#official IEBC 2019 Boundaries (extraccted from the OCHA geodatabase)
counties = pd.read_csv("data/kenya_counties.csv")  #47 counties
subs = pd.read_csv("data/kenya_subcounties.csv")    #290 sub-counties


def norm(name):
    # Reduce a name to lowercase letters only, so that
    #Homa Bay", "HOMABAY" and "Homa0Bay" all resolve as "homabay".
    #This basically fixes most spelling diffs between the sources
    return re.sub(r"[^a-z]", "", str(name).lower())


#county name -> county code, e.g. "kisumu" -> "KE042"
county_code = {norm(n): c for n, c in zip(counties.admin1Name_en, counties.admin1Pcode)}

#sub-county name -> its parent couonty code e.g "kinango" -> "KE002" (Kwale).
#it lets us place records that name a subcounty instead of a county
sub_to_county = {norm(n): c for n, c in zip(subs.admin2Name_en, subs.admin1Pcode)}

#Names that normalising can't fix: old pre-2013 district names, towns,
#rivers and a typo, each map to the county it falls in.

ALIASES = {
    #Elgeyo-Marakwet was two districts(Kiyo, Matakwet) before 2013
    "keiyo": "KE028", "marakwet": "KE028", "elgeyo":"KE028", "keiyomarakwet": "KE028",
    #Turkana: typo in EM-DAT, plus towns within the county
    "tarkana": "KE023", "lodwar": "KE023", "katilu" : "KE023",
    "rongai": "KE032", "naivasha": "KE032",                        #Nakuru
    "thika": "KE022",                                              #Kiambu
    "maragua": "KE021",                                            #Murang'a
    "nyando": "KE042", "kisimu": "KE042",                          #Kisumu
    "rachuonyo": "KE043", "suba": "KE043",                         #Homa Bay
    "koibatek": "KE030", "marigat": "KE030", "salabani": "KE030",  #Baringo
    "mtelgon": "KE039",                                            #Bungoma
    "malindi" : "KE003",                                           #Kilifi
    "moyale" : "KE010", "chalbi": "KE010",                         #Marsabit
    "garsen" : "KE004", "bura": "KE004", "madogo": "KE004",
    "bangale": "KE004", "tanariverdelta": "KE004",                 #Tana River
    "ijara" : "KE007", "dadaab" : "KE007",                         #Garissa
    "taveta": "KE006",                                             #Taita Taveta
    "kitengela": "KE034", "ngong": "KE034",                        #Kajiado
    "centralkisii":"KE045",                                        #Kisii
    "nandinorth": "KE029", "nandisouth": "KE029",                  #Nandi
    "athiriver": "KE016",
}

#Pre-2013 PRONVINCES. Each covers many counties("Rift Valley" = 14),
#so we cant tell which county flooded. DECISION: ignore province-only
#mentions rather than label every county inside them- labelling all
#14 Rift Vallry counties would invent far more floods than happened.
PROVINCES = {"coast", "coastal", "nyanza", "western", "riftvalley", "central",
              "eastern", "northeastern", "northwestern", "nairobi"}


def to_pcode(name):
    #Tries in the order of known alias, county name and sub-county name
    key = norm(name)
    if key in ALIASES:
        return ALIASES[key]
    if key in county_code:
        return county_code[key]
    return sub_to_county.get(key) #if nothing matches


#Filler word in EM-DAT's location text that aren't part of any name
JUNK = (r"\b(counties|county|near|region|regions|and|the|areas?|districts?|provinces?|"
        r"town|city|villages?|camp)\b")

def clean(part):
    return re.sub(JUNK, "", part, flags=re.I).strip(" ()")


def counties_in(location):
    #Turn one EM-DAT location string into a set of county codes.
    #an example is "Athi River (Machakos" -> tries "Athi River" AND "Machakos",
    #since older records put the county inside brackets.
    found, unmatched = set(), []
    for part in re.split(r"[,;/]| and ", location):
        pieces = [p for p in part.split("(") if clean(p)]
        codes = {to_pcode(clean(p)) for p in pieces} - {None}
        if codes :
            found |= codes
        elif norm (clean(part.split("(")[0])) not in PROVINCES and clean(part):
            unmatched.append(clean(part))
    return found, unmatched


if __name__ == "__main__":
    # Only runs when this file is executed directly,
    # NOT when make_labels.py imports functions from it.

    #---checking source 1, DesInventar (1 county per row)
    di = pd.read_csv("data/DesInventar_Geocoded.csv", low_memory=False)
    di = di[di.Country == "Kenya"]
    print("DesInventar unmatched:", sorted({n for n in di.Adm1.dropna() if to_pcode(n) is None}))

    #---checking source 2: EM-DAT (free-text list of places)
    em = pd.read_csv("data/EMDAT_Geocoded.csv", low_memory=False)
    em = em[(em.iso == "KEN") & (em["start year"] >= 2002)] #the labelled window

    bad = set()
    for loc in em.location.dropna():
        bad |= set(counties_in(loc)[1])
    print("EM-DAT unmatched names:", sorted(bad))

    #Events where NO county could be identified at all, they'll be lost
    lost = [(d, l) for d, l in zip(em["disno."], em.location)
            if isinstance(l, str) and not counties_in(l)[0]]
    print("EM-DAT events with no county found:", lost)
    print("EM-DAT events with no location at all:", em.location.isna().sum())

# Run the matching checks
di = pd.read_csv("data/DesInventar_Geocoded.csv", low_memory=False)
di = di[di.Country == "Kenya"]
print("DesInventar unmatched:", sorted({n for n in di.Adm1.dropna() if to_pcode(n) is None}))
em = pd.read_csv("data/EMDAT_Geocoded.csv", low_memory=False)
em = em[(em.iso == "KEN") & (em["start year"] >= 2002)]
bad = set()
for loc in em.location.dropna():
    bad |= set(counties_in(loc)[1])
print("EM-DAT unmatched names:", sorted(bad))

DesInventar unmatched: []
EM-DAT unmatched names: ['Bukhay', 'Kalifonia', 'Napetet', 'Ngambo', 'Soweto']
EM-DAT events with no county found: [('2006-0234-KEN', 'Nyanza, Coast, Western provinces')]
EM-DAT events with no location at all: 1


DesInventar unmatched: []
EM-DAT unmatched names: ['Bukhay', 'Kalifonia', 'Napetet', 'Ngambo', 'Soweto']


### 3.2 Building the labels

In [3]:
import calendar
import pandas as pd
from county_match import to_pcode, counties_in


#-------------------------------------------------------------
#PURPOSE
#Turn flood RECORDS ("KKisumu flooded 12-20 April 2013") into a
#LABEL for every (sub-county, dekad) row in the rainfall data:
#  flooded = 1 if any source reports a flood touching that dekad
#  flooded = 0 otherwise
#Output: data/labels.csv, one row per sub-county per dekad.
#----------------------------------------------------------------


#labelled window: our flood sources are reliable from 2002
#and EM-DAT's coverage of Kenya ends in Dec-2024
WINDOW_START = pd.Timestamp("2002-01-01")
WINDOW_END = pd.Timestamp("2024-12-31")

#Some records span 60-120 dats - that describes a whole rainsy season.
#not 4 months of continuous flooding. Cap how long one record can
#label fo. None = no cap. (Decision = see the cap comparison printed below)
MAX_EVENT_DAYS = 60

#One EM-DAT event has no end month. DFO records the same event
#(DFO #5477) as ending 11 Nov 2023, so we borrow that date.
END_OVERRIDES = {"2023-0721-KEN": pd.Timestamp("2023-11-11")}


#-----dekad grid from rainfall data-------
rain = pd.read_csv("data/ken-rainfall-subnat-full.csv", parse_dates=["date"])
rain = rain[(rain.adm_level == 2)                       #sub-counties only
            & rain.date.between(WINDOW_START, WINDOW_END)]
dekads = sorted(rain.date.unique())
subcounties = sorted(rain.PCODE.unique())


def dekad_start(ts):
    #Snap any date to the start of its dekad: 1st, 11th or 21st.
    return ts.replace(day=1 if ts.day <= 10 else 11 if ts.day <= 20 else 21)


def month_end(year, month):
    return pd.Timestamp(year, month, calendar.monthrange(year, month)[1])


def month_end(year, month):
    return pd.Timestamp(year, month, calendar.monthrange(year, month)[1])


#Each record becomes: (county code, start date, end date, source)
records = []

# ---------- Source 1: DesInventar - one county, one date per row ----------
di = pd.read_csv("data/DesInventar_Geocoded.csv", low_memory=False)
di = di[di.Country == "Kenya"]
for _, r in di.iterrows():
    county = to_pcode(r.Adm1)
    date = pd.to_datetime(r.Date, errors="coerce")
    if pd.isna(date):
        # One record reads "2016/03/00" - day unknown, so it covers the month
        y, m, _ = map(int, str(r.Date).split("/"))
        start, end = pd.Timestamp(y, m, 1), month_end(y, m)
    else:
        start = date
        end = date + pd.Timedelta(days=int(r["Duration (d)"]))
    records.append((county, start, end, "DesInventar"))
 
# ---------- Source 2: EM-DAT - several counties per event ----------
em = pd.read_csv("data/EMDAT_Geocoded.csv", low_memory=False)
em = em[(em.iso == "KEN") & (em["start year"] >= 2002)].dropna(subset=["location"])
for _, r in em.iterrows():
    sy, sm = int(r["start year"]), int(r["start month"])
    # Missing day -> assume the event spans the whole month (2 events)
    start = pd.Timestamp(sy, sm, int(r["start day"]) if pd.notna(r["start day"]) else 1)
    if r["disno."] in END_OVERRIDES:
        end = END_OVERRIDES[r["disno."]]
    else:
        ey, em_ = int(r["end year"]), int(r["end month"])
        end = (pd.Timestamp(ey, em_, int(r["end day"])) if pd.notna(r["end day"])
               else month_end(ey, em_))
    for county in counties_in(r.location)[0]:
        records.append((county, start, end, "EM-DAT"))
 
events = pd.DataFrame(records, columns=["county", "start", "end", "source"])
events["days"] = (events.end - events.start).dt.days
 
 
def build_labels(cap):
    # Expand every record into (sub-county, dekad) pairs it touches.
    hits = []
    for e in events.itertuples():
        end = e.end if cap is None else min(e.end, e.start + pd.Timedelta(days=cap))
        first, last = dekad_start(e.start), dekad_start(end)
        touched = [d for d in dekads if first <= d <= last]
        # A county-level record labels EVERY sampled sub-county in that county
        for pcode in subcounties:
            if pcode[:5] == e.county:
                hits += [(pcode, d, e.source) for d in touched]
    hits = pd.DataFrame(hits, columns=["PCODE", "date", "source"])
 
    # Union across sources: one flood row per (sub-county, dekad), however
    # many records mention it. This is what makes duplicates harmless.
    pos = (hits.groupby(["PCODE", "date"])["source"]
               .agg(lambda s: ";".join(sorted(set(s)))).reset_index())
 
    grid = pd.MultiIndex.from_product([subcounties, dekads],
                                      names=["PCODE", "date"]).to_frame(index=False)
    labels = grid.merge(pos, on=["PCODE", "date"], how="left")
    labels["flooded"] = labels.source.notna().astype(int)
    return labels
 
 
# ---------- Decide the cap with evidence, not a guess ----------
print("records:", len(events), "| by source:", events.source.value_counts().to_dict())
print("records longer than 30 days:", (events.days > 30).sum())
for cap in [None, 60, 30]:
    lab = build_labels(cap)
    print(f"cap={cap}: {lab.flooded.sum()} positive rows of {len(lab)} "
          f"({100 * lab.flooded.mean():.2f}%)")
 
labels = build_labels(MAX_EVENT_DAYS)
labels.to_csv("data/labels.csv", index=False)
print("\nsaved data/labels.csv with cap =", MAX_EVENT_DAYS)
 
# ---------- Sanity check: MAM 2024 in Nairobi (Lang'ata) ----------
check = labels[(labels.PCODE == "KE047277")
               & labels.date.between("2024-03-01", "2024-05-31")]
print("\nNairobi MAM 2024:")
print(check.to_string(index=False))

records: 763 | by source: {'DesInventar': 449, 'EM-DAT': 314}
records longer than 30 days: 160
cap=None: 2670 positive rows of 60444 (4.42%)


cap=60: 2557 positive rows of 60444 (4.23%)
cap=30: 2072 positive rows of 60444 (3.43%)



saved data/labels.csv with cap = 60

Nairobi MAM 2024:
   PCODE       date source  flooded
KE047277 2024-03-01    NaN        0
KE047277 2024-03-11    NaN        0
KE047277 2024-03-21 EM-DAT        1
KE047277 2024-04-01 EM-DAT        1
KE047277 2024-04-11 EM-DAT        1
KE047277 2024-04-21 EM-DAT        1
KE047277 2024-05-01 EM-DAT        1
KE047277 2024-05-11    NaN        0
KE047277 2024-05-21    NaN        0


## 4. Features

In [4]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# The purpose of this is to:
# Build the model's inputs (features) for every (sub-county, dekad)
# row, then join them to the flood labels.
# it'll Output: data/features.csv - the final modelling table.
#
# The task is a NOWCAST: "given the rain observed up to and including
# this dekad, is this sub-county flooding?" Every feature only uses
# information from this dekad or earlier - never the future.
# ---------------------------------------------------------------

rain = pd.read_csv("data/ken-rainfall-subnat-full.csv", parse_dates=["date"])
rain = rain[rain.adm_level == 2].copy()          # sub-counties only
rain = rain.sort_values(["PCODE", "date"])       # required for lags to be correct

# ---------- 1. Features the dataset already gives us ----------
# rfh      rainfall this dekad (mm)
# rfh_avg  long-term average for this dekad here, how wet that place normally is
# rfq      this dekad as % of normal
# r1h/r3h  rainfall over the last 1 and 3 months - ANTECEDENT rain.
#          Saturated ground floods on less rain, so this captures soil
#          wetness, which we can't observe directly.
# r1q/r3q  those as % of normal
BASE = ["rfh", "rfh_avg", "rfq", "r1h", "r3h", "r1q", "r3q"]

# ---------- 2. Lags: what happened in the previous dekads ----------
# Computed per sub-county (groupby) so one place's rain never leaks
# into another's row. Computed on the FULL 1981+ series before we cut
# to 2002, so the very first 2002 rows still have a previous dekad.
g = rain.groupby("PCODE")
rain["rfh_lag1"] = g.rfh.shift(1)
rain["rfh_lag2"] = g.rfh.shift(2)
rain["rfq_lag1"] = g.rfq.shift(1)

# ---------- 3. Season, as a cycle ----------
# Dekad of year runs 1..36. Plain numbers would tell the model dekad 36
# (late Dec) is far from dekad 1 (early Jan) - they're neighbours.
# Sine/cosine wrap the year into a circle so Dec sits next to Jan.
dekad_of_year = (rain.date.dt.month - 1) * 3 + (rain.date.dt.day // 10).clip(upper=2) + 1
rain["season_sin"] = np.sin(2 * np.pi * dekad_of_year / 36)
rain["season_cos"] = np.cos(2 * np.pi * dekad_of_year / 36)

# ---------- 4. How widespread is the wet spell? ----------
# Share of all 73 sub-counties above 200% of normal this dekad.
# A storm hitting one place vs a country-wide El Nino season look
# identical to a single sub-county's rain gauge - this tells them apart.
rain["nat_frac_anom"] = rain.groupby("date").rfq.transform(lambda s: (s >= 200).mean())

FEATURES = BASE + ["rfh_lag1", "rfh_lag2", "rfq_lag1",
                   "season_sin", "season_cos", "nat_frac_anom"]

# DELIBERATELY EXCLUDED:
#  - year:  flood REPORTING collapsed after 2016, so year would teach
#           the model reporting history, not flooding.
#  - PCODE: would let the model memorise which counties report floods
#           most often, rather than learn from rainfall.

# ---------- Join to labels ----------
labels = pd.read_csv("data/labels.csv", parse_dates=["date"])
df = labels.merge(rain[["PCODE", "date"] + FEATURES], on=["PCODE", "date"], how="left")

print("rows:", len(df), "| positives:", df.flooded.sum(),
      f"({100 * df.flooded.mean():.2f}%)")
print("missing values per feature (should all be 0):")
print(df[FEATURES].isna().sum().to_string())

# Quick sanity check - flooded dekads should look wetter on every rain feature
print("\nmedian feature value, flooded vs not:")
print(df.groupby("flooded")[FEATURES].median().T.round(2).to_string())

df[["PCODE", "date", "flooded", "source"] + FEATURES].to_csv("data/features.csv", index=False)
print("\nsaved data/features.csv")

rows: 60444 | positives: 2557 (4.23%)
missing values per feature (should all be 0):
rfh              0
rfh_avg          0
rfq              0
r1h              0
r3h              0
r1q              0
r3q              0
rfh_lag1         0
rfh_lag2         0
rfq_lag1         0
season_sin       0
season_cos       0
nat_frac_anom    0

median feature value, flooded vs not:
flooded             0       1
rfh             20.96   51.95
rfh_avg         27.60   40.23
rfq             91.44  128.38
r1h             77.46  173.06
r3h            271.49  364.09
r1q             95.22  140.24
r3q             99.57  130.94
rfh_lag1        20.90   53.56
rfh_lag2        21.06   46.50
rfq_lag1        91.23  131.17
season_sin      -0.00    0.50
season_cos      -0.00    0.00
nat_frac_anom    0.00    0.10



saved data/features.csv


## 5. Base model: logistic regression

In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_score, recall_score, f1_score, confusion_matrix)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------
# PURPOSE
# Train the BASE MODEL - L2-regularised logistic regression - and
# compare it against the 50mm/200% threshold rule on unseen years.
# Output: data/predictions.csv in the agreed team format.
# ---------------------------------------------------------------

FEATURES = ["rfh", "rfh_avg", "rfq", "r1h", "r3h", "r1q", "r3q",
            "rfh_lag1", "rfh_lag2", "rfq_lag1",
            "season_sin", "season_cos", "nat_frac_anom"]
SEED = 42   # fixed so every result in the notebook is reproducible

df = pd.read_csv("data/features.csv", parse_dates=["date"]).sort_values(["date", "PCODE"])

# ---------- Time-based split ----------
# Train on 2002-2019, test on 2020-2024. The test years are never
# touched until the very end - not for tuning, not for anything.
df["split"] = np.where(df.date >= "2020-01-01", "test", "train")
train, test = df[df.split == "train"], df[df.split == "test"]
X_train, y_train = train[FEATURES], train.flooded
X_test, y_test = test[FEATURES], test.flooded
print(f"train: {len(train)} rows, {y_train.sum()} floods | "
      f"test: {len(test)} rows, {y_test.sum()} floods")


def make_model(C):
    # StandardScaler first: features are on wildly different scales
    # (mm of rain vs a 0-1 fraction), and regularisation penalises
    # coefficients by size, so unscaled features would be punished unfairly.
    # class_weight="balanced": floods are ~4% of rows; without this the
    # model can score 96% by always saying "no flood".
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, class_weight="balanced", max_iter=1000, random_state=SEED),
    )


# ---------- Cross-validation to choose regularisation strength ----------
# C controls L2 regularisation: SMALL C = strong penalty = simpler model.
# TimeSeriesSplit always validates on a period AFTER its training period.
# We split on whole dekads (not rows) so all 73 sub-counties of one dekad
# land on the same side - otherwise the same storm sits in both halves.
dates = np.sort(train.date.unique())
tscv = TimeSeriesSplit(n_splits=5)

# Score = average precision (area under the precision-recall curve).
# Not accuracy: with 4% floods, accuracy rewards ignoring floods.
print("\nC        mean PR-AUC across 5 time folds")
cv_scores = {}
for C in [0.001, 0.01, 0.1, 1, 10]:
    fold_scores = []
    for tr_idx, va_idx in tscv.split(dates):
        tr = train[train.date.isin(dates[tr_idx])]
        va = train[train.date.isin(dates[va_idx])]
        m = make_model(C).fit(tr[FEATURES], tr.flooded)
        fold_scores.append(average_precision_score(va.flooded, m.predict_proba(va[FEATURES])[:, 1]))
    cv_scores[C] = np.mean(fold_scores)
    print(f"{C:<8} {cv_scores[C]:.3f}   (folds: {', '.join(f'{s:.2f}' for s in fold_scores)})")

best_C = max(cv_scores, key=cv_scores.get)
print("chosen C:", best_C)

# ---------- Final model: all training years, chosen C ----------
model = make_model(best_C).fit(X_train, y_train)
test_prob = model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

# ---------- The baseline rule on the same test rows ----------
rule_pred = ((test.rfh >= 50) & (test.rfq >= 200)).astype(int)


def report(name, y, pred, prob=None):
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    line = (f"{name:<14} precision {precision_score(y, pred):.3f}  "
            f"recall {recall_score(y, pred):.3f}  F1 {f1_score(y, pred):.3f}")
    if prob is not None:
        line += f"  ROC-AUC {roc_auc_score(y, prob):.3f}  PR-AUC {average_precision_score(y, prob):.3f}"
    print(line)
    print(f"{'':<14} caught {tp} floods, missed {fn}, false alarms {fp}")


print("\n=== TEST SET 2020-2024 (never seen during training or tuning) ===")
print(f"base rate: {y_test.mean():.3f}  <- a random guesser's PR-AUC")
report("50mm rule", y_test, rule_pred)
report("logreg @0.5", y_test, test_pred, test_prob)

# ---------- What did the model learn? ----------
# Features were standardised, so coefficient sizes are comparable:
# bigger = more influence. Positive = pushes towards "flood".
coefs = pd.Series(model[-1].coef_[0], index=FEATURES).sort_values(key=abs, ascending=False)
print("\ncoefficients (standardised):")
print(coefs.round(3).to_string())

# ---------- Save predictions in the agreed team format ----------
out = df[["PCODE", "date", "split", "flooded"]].rename(columns={"flooded": "y_true"})
out["rule_pred"] = ((df.rfh >= 50) & (df.rfq >= 200)).astype(int)
out["logreg_prob"] = model.predict_proba(df[FEATURES])[:, 1]
out.to_csv("data/predictions.csv", index=False)
print("\nsaved data/predictions.csv")

train: 47304 rows, 2054 floods | test: 13140 rows, 503 floods

C        mean PR-AUC across 5 time folds


0.001    0.177   (folds: 0.09, 0.19, 0.23, 0.08, 0.31)
0.01     0.177   (folds: 0.09, 0.19, 0.22, 0.08, 0.31)


0.1      0.176   (folds: 0.08, 0.19, 0.22, 0.08, 0.31)
1        0.175   (folds: 0.08, 0.18, 0.22, 0.08, 0.31)


10       0.175   (folds: 0.08, 0.18, 0.22, 0.08, 0.31)
chosen C: 0.001

=== TEST SET 2020-2024 (never seen during training or tuning) ===
base rate: 0.038  <- a random guesser's PR-AUC
50mm rule      precision 0.107  recall 0.221  F1 0.144
               caught 111 floods, missed 392, false alarms 929
logreg @0.5    precision 0.079  recall 0.722  F1 0.142  ROC-AUC 0.757  PR-AUC 0.119
               caught 363 floods, missed 140, false alarms 4235

coefficients (standardised):
r3q              0.492
season_sin       0.414
r3h             -0.367
nat_frac_anom    0.332
season_cos       0.301
rfh_lag2         0.287
r1h              0.215
rfh_lag1         0.197
r1q              0.182
rfh_avg          0.155
rfq             -0.076
rfq_lag1         0.064
rfh              0.039

saved data/predictions.csv


## 6. Comparison model: random forest

In [6]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
from sklearn.model_selection import TimeSeriesSplit

# ---------------------------------------------------------------
# PURPOSE
# Train the COMPARISON MODEL - a random forest - with exactly the same
# data, split and cross-validation as the logistic regression, so any
# difference in results comes from the model, not the setup.
# Adds an rf_prob column to data/predictions.csv.
# ---------------------------------------------------------------

FEATURES = ["rfh", "rfh_avg", "rfq", "r1h", "r3h", "r1q", "r3q",
            "rfh_lag1", "rfh_lag2", "rfq_lag1",
            "season_sin", "season_cos", "nat_frac_anom"]
SEED = 42

df = pd.read_csv("data/features.csv", parse_dates=["date"]).sort_values(["date", "PCODE"])
df["split"] = np.where(df.date >= "2020-01-01", "test", "train")
train, test = df[df.split == "train"], df[df.split == "test"]


def make_model(min_leaf):
    # A random forest = many decision trees, each trained on a random
    # sample of rows and features, voting together. Unlike logistic
    # regression it can learn interactions ("heavy rain matters MORE
    # when the ground is already wet") without us coding them.
    # min_samples_leaf is its main brake on overfitting: each final
    # decision must be based on at least this many rows.
    # No scaling needed - trees split on thresholds, so units don't matter.
    return RandomForestClassifier(
        n_estimators=300, min_samples_leaf=min_leaf, max_features="sqrt",
        class_weight="balanced", n_jobs=-1, random_state=SEED,
    )


# ---------- Same time-series CV as logreg, tuning the overfitting brake ----------
dates = np.sort(train.date.unique())
tscv = TimeSeriesSplit(n_splits=5)

print("min_leaf  mean PR-AUC across 5 time folds")
cv_scores = {}
for leaf in [5, 20, 50, 100, 200]:
    fold_scores = []
    for tr_idx, va_idx in tscv.split(dates):
        tr = train[train.date.isin(dates[tr_idx])]
        va = train[train.date.isin(dates[va_idx])]
        m = make_model(leaf).fit(tr[FEATURES], tr.flooded)
        fold_scores.append(average_precision_score(va.flooded, m.predict_proba(va[FEATURES])[:, 1]))
    cv_scores[leaf] = np.mean(fold_scores)
    print(f"{leaf:<9} {cv_scores[leaf]:.3f}   (folds: {', '.join(f'{s:.2f}' for s in fold_scores)})")

best_leaf = max(cv_scores, key=cv_scores.get)
print("chosen min_samples_leaf:", best_leaf)

model = make_model(best_leaf).fit(train[FEATURES], train.flooded)
test_prob = model.predict_proba(test[FEATURES])[:, 1]

# ---------- Test results, on the same terms as logreg ----------
print("\n=== TEST SET 2020-2024 ===")
print(f"ROC-AUC {roc_auc_score(test.flooded, test_prob):.3f}  "
      f"PR-AUC {average_precision_score(test.flooded, test_prob):.3f}  "
      f"(base rate {test.flooded.mean():.3f})")

# Fair comparison with the rule: what precision at the rule's recall,
# and what recall at the rule's precision?
p, r, _ = precision_recall_curve(test.flooded, test_prob)
rule = (test.rfh >= 50) & (test.rfq >= 200)
rule_p = test.flooded[rule].mean()
rule_r = rule[test.flooded == 1].mean()
print(f"rule:            precision {rule_p:.3f} at recall {rule_r:.3f}")
# precision_recall_curve lists recall from high to low; the LAST point
# still at or above the rule's recall is the RF's best precision there
same_recall_precision = p[np.where(r >= rule_r)[0][-1]]
same_precision_recall = r[p >= rule_p].max()
print(f"RF at the rule's recall:    precision {same_recall_precision:.3f}")
print(f"RF at the rule's precision: recall {same_precision_recall:.3f}")

# ---------- What does the forest rely on? ----------
# Importance = how much each feature reduces uncertainty across all splits.
# Always positive - it says HOW MUCH a feature matters, not which direction.
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nfeature importance:")
print(imp.round(3).to_string())

# ---------- Add to the shared predictions file ----------
pred = pd.read_csv("data/predictions.csv", parse_dates=["date"])
pred = pred.drop(columns=["rf_prob"], errors="ignore")
rf_all = df[["PCODE", "date"]].assign(rf_prob=model.predict_proba(df[FEATURES])[:, 1])
pred = pred.merge(rf_all, on=["PCODE", "date"], how="left")
pred.to_csv("data/predictions.csv", index=False)
print("\nadded rf_prob to data/predictions.csv")

min_leaf  mean PR-AUC across 5 time folds


5         0.127   (folds: 0.06, 0.16, 0.16, 0.11, 0.14)


20        0.129   (folds: 0.06, 0.16, 0.17, 0.11, 0.14)


50        0.135   (folds: 0.05, 0.15, 0.19, 0.11, 0.16)


100       0.137   (folds: 0.05, 0.15, 0.20, 0.10, 0.19)


200       0.150   (folds: 0.05, 0.15, 0.20, 0.10, 0.26)
chosen min_samples_leaf: 200



=== TEST SET 2020-2024 ===
ROC-AUC 0.769  PR-AUC 0.103  (base rate 0.038)
rule:            precision 0.107 at recall 0.221
RF at the rule's recall:    precision 0.109
RF at the rule's precision: recall 0.469

feature importance:
r1q              0.191
r3q              0.166
r1h              0.119
nat_frac_anom    0.095
season_sin       0.089
season_cos       0.074
rfq_lag1         0.065
rfh_lag1         0.058
rfh_lag2         0.040
rfq              0.033
rfh              0.033
rfh_avg          0.020
r3h              0.015



added rf_prob to data/predictions.csv


## 7. Model comparison (Section B)

*Eric's evaluation harness: `evaluation.py`.*

In [7]:
%run evaluation.py --input data/predictions.csv --features data/features.csv --split test

              Model  Precision  Recall    F1  ROC-AUC  TP   FP  FN   TN
Logistic Regression      0.079   0.722 0.142    0.757 363 4235 140 8402
      Random Forest      0.082   0.757 0.148    0.769 381 4275 122 8362
         50 mm Rule      0.087   0.614 0.152    0.755 309 3248 194 9389

Confusion matrices (rows = actual [no flood, flood]; columns = predicted [no flood, flood]):
Logistic Regression: [[8402, 4235], [140, 363]]
Random Forest: [[8362, 4275], [122, 381]]
50 mm Rule: [[9389, 3248], [194, 309]]


## 8. Alert threshold and error analysis

In [8]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------
# PURPOSE
# 1. CHOOSE the alert threshold for the base model - using training
#    years only, so the test years stay untouched.
# 2. ERROR ANALYSIS - on the test years, look at which floods the
#    model missed and where its false alarms come from.
# ---------------------------------------------------------------

FEATURES = ["rfh", "rfh_avg", "rfq", "r1h", "r3h", "r1q", "r3q",
            "rfh_lag1", "rfh_lag2", "rfq_lag1",
            "season_sin", "season_cos", "nat_frac_anom"]
SEED, C = 42, 0.001          # same settings train_logreg.py chose

df = pd.read_csv("data/features.csv", parse_dates=["date"]).sort_values(["date", "PCODE"])
pred = pd.read_csv("data/predictions.csv", parse_dates=["date"])
df = df.merge(pred[["PCODE", "date", "split", "logreg_prob", "rule_pred"]], on=["PCODE", "date"])
train, test = df[df.split == "train"], df[df.split == "test"]


def make_model():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=C, class_weight="balanced",
                                            max_iter=1000, random_state=SEED))


# ================= PART 1: choose the threshold =================
# The threshold turns a probability into an alert. Picking it by looking
# at TEST results would be cheating - we'd be tuning on the exam paper.
# So we get "out-of-fold" probabilities on the TRAINING years: each
# validation period is scored by a model that never saw it.
dates = np.sort(train.date.unique())
oof = []
for tr_idx, va_idx in TimeSeriesSplit(n_splits=5).split(dates):
    tr = train[train.date.isin(dates[tr_idx])]
    va = train[train.date.isin(dates[va_idx])]
    m = make_model().fit(tr[FEATURES], tr.flooded)
    oof.append(pd.DataFrame({"y": va.flooded, "p": m.predict_proba(va[FEATURES])[:, 1],
                             "rule": ((va.rfh >= 50) & (va.rfq >= 200)).astype(int)}))
oof = pd.concat(oof)

# RULE FOR THE THRESHOLD: catch as many floods as possible (recall first,
# because a missed flood strands a responder) WITHOUT being less precise
# than the rule already in use. "At least as trustworthy as today's
# alerts, and catches more."
rule_precision = precision_score(oof.y, oof.rule)
p, r, th = precision_recall_curve(oof.y, oof.p)
ok = np.where(p[:-1] >= rule_precision)[0]          # thresholds meeting the bar
best = ok[np.argmax(r[ok])]                         # ...with the highest recall
threshold = th[best]
print(f"rule's precision on training years: {rule_precision:.3f}")
print(f"chosen threshold: {threshold:.3f}  "
      f"(training-years recall {r[best]:.3f}, precision {p[best]:.3f})")

# ================= PART 2: does it hold on the unseen years? =================
test = test.copy()
test["alert"] = (test.logreg_prob >= threshold).astype(int)
print("\n=== TEST 2020-2024 at the chosen threshold ===")
for name, col in [("50mm rule", "rule_pred"), ("logreg", "alert")]:
    print(f"{name:<10} precision {precision_score(test.flooded, test[col]):.3f}  "
          f"recall {recall_score(test.flooded, test[col]):.3f}  "
          f"alerts {test[col].sum()}  floods caught {(test[col] & test.flooded).sum()}")

# ================= PART 3: error analysis =================
floods = test[test.flooded == 1]
caught, missed = floods[floods.alert == 1], floods[floods.alert == 0]

print(f"\n--- MISSED floods ({len(missed)}) vs CAUGHT floods ({len(caught)}) ---")
print("median values:")
cols = ["rfh", "rfq", "r1q", "r3q", "nat_frac_anom"]
print(pd.DataFrame({"caught": caught[cols].median(), "missed": missed[cols].median()}).round(2).to_string())

print("\nshare of floods missed, by label source:")
print(floods.groupby("source").alert.apply(lambda a: f"{1 - a.mean():.0%} missed of {len(a)}").to_string())

print("\nshare of floods missed, by month:")
print(floods.groupby(floods.date.dt.month).alert
      .apply(lambda a: f"{1 - a.mean():.0%} of {len(a)}").to_string())

# False alarms: alerts where no flood was recorded. Are they random,
# or clustered right next to real floods (same sub-county, adjacent dekad)?
test = test.sort_values(["PCODE", "date"])
g = test.groupby("PCODE").flooded
test["near_flood"] = ((g.shift(1) == 1) | (g.shift(-1) == 1)).astype(int)
fa = test[(test.alert == 1) & (test.flooded == 0)]
print(f"\n--- FALSE ALARMS ({len(fa)}) ---")
print(f"right next to a recorded flood (dekad before/after): {fa.near_flood.mean():.0%}")
print(f"during a country-wide wet spell (>=10% of sub-counties anomalous): "
      f"{(fa.nat_frac_anom >= 0.10).mean():.0%}")
print("false alarms by year:")
print(fa.date.dt.year.value_counts().sort_index().to_string())

# ---------- Where are floods missed most? ----------
# Missed floods had near-normal LOCAL rain. Check whether that's
# concentrated in counties that flood from rain falling elsewhere:
# Tana River and Garissa sit downstream on the Tana River, which is fed
# by rain in the central highlands (Mt Kenya / Aberdares), not locally.
names = pd.read_csv("data/kenya_counties.csv").set_index("admin1Pcode").admin1Name_en
floods = floods.assign(county=floods.PCODE.str[:5].map(names))
print("\nmiss rate by county (counties with the most test floods):")
by_county = floods.groupby("county").alert.agg(floods="size", missed=lambda a: 1 - a.mean())
print(by_county.sort_values("floods", ascending=False).head(10).round(2).to_string())

downstream = floods.county.isin(["Tana River", "Garissa"])
print(f"\nTana River + Garissa: {1 - floods[downstream].alert.mean():.0%} missed "
      f"| all other counties: {1 - floods[~downstream].alert.mean():.0%} missed")

rule's precision on training years: 0.170
chosen threshold: 0.669  (training-years recall 0.514, precision 0.170)

=== TEST 2020-2024 at the chosen threshold ===
50mm rule  precision 0.107  recall 0.221  alerts 1040  floods caught 111
logreg     precision 0.096  recall 0.559  alerts 2930  floods caught 281

--- MISSED floods (222) vs CAUGHT floods (281) ---
median values:
               caught  missed
rfh             97.10   34.21
rfq            170.40   95.69
r1q            158.19   95.90
r3q            152.87  109.38
nat_frac_anom    0.36    0.00

share of floods missed, by label source:
source
DesInventar            50% missed of 18
DesInventar;EM-DAT      20% missed of 5
EM-DAT                44% missed of 480

share of floods missed, by month:
date
3      46% of 54
4     38% of 199
5     44% of 174
6     100% of 25
10     67% of 24
11      8% of 24
12      67% of 3

--- FALSE ALARMS (2649) ---
right next to a recorded flood (dekad before/after): 4%
during a country-wide wet spell 